# Alineación Pythia ↔ humano (AoA)

Join entre `step_stabilize` por paradigma (definido en `docs/04_experimental_design.md`) y la tabla `docs/05_human_alignment.md` / `data/human_milestones.csv`. Spearman + bootstrap.

In [ ]:
from pathlib import Path

import pandas as pd

from ontogenia.human_alignment import run_human_alignment

ROOT = Path("..")
PARQUET = ROOT / "results" / "aggregated_metrics.parquet"
AOA = ROOT / "data" / "human_milestones.csv"
PARQUET.exists(), AOA.exists()

In [ ]:
if not PARQUET.exists() or not AOA.exists():
    raise FileNotFoundError(
        "Falta parquet o CSV AoA. Requisitos:\n"
        "- python -m ontogenia aggregate --output-parquet results/aggregated_metrics.parquet\n"
        "- data/human_milestones.csv con columnas: task, aoa_months"
    )

result = run_human_alignment(
    metrics_parquet=PARQUET,
    aoa_csv=AOA,
    metric_col="acc,none",
    target_model_size="160m",  # cambiar a 14m / 410m según análisis
)

result.overlap[["task", "training_step_stabilize", "aoa_months"]].head(), result.stats

In [ ]:
# Guardar outputs para paper / tracking
stats_path = ROOT / "results" / "human_alignment_stats.json"
overlap_path = ROOT / "results" / "human_alignment_overlap.parquet"

result.overlap.to_parquet(overlap_path, index=False)
stats_path.write_text(result.stats.to_json(orient="records", indent=2), encoding="utf-8")

stats_path, overlap_path